# 🖥️ Running Local LLMs: The Complete Deep Dive

## 🎯 From Workshop 2 → Advanced Local Model Mastery

---

### 📋 What You'll Master in This Notebook

| Section | Topic | Difficulty |
|---------|-------|------------|
| 1️⃣ | Understanding Local Models | ⭐ Beginner |
| 2️⃣ | Ollama Deep Dive & Python Integration | ⭐⭐ Intermediate |
| 3️⃣ | HuggingFace Transformers Local | ⭐⭐ Intermediate |
| 4️⃣ | llama.cpp & GGUF Models | ⭐⭐⭐ Advanced |
| 5️⃣ | Performance Optimization | ⭐⭐⭐ Advanced |
| 6️⃣ | Building Production Applications | ⭐⭐⭐ Advanced |

---

### 🔗 Prerequisites

- Completed Workshop 2 basics
- Python 3.8+ installed locally
- Basic understanding of APIs and REST
- For full hands-on: Local machine with 8GB+ RAM (16GB+ recommended)

> **💡 Note**: While this notebook includes code that runs locally, we'll also demonstrate API patterns that you can test in Colab when connecting to a local Ollama server.

---

# 📚 Section 1: Understanding Local LLMs

## 1.1 Why Run Models Locally?

### The Privacy vs Performance Trade-off

```
┌─────────────────────────────────────────────────────────────────────┐
│                    CLOUD vs LOCAL COMPARISON                        │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  ☁️ CLOUD APIs (OpenAI, Anthropic, Google)                          │
│  ├─ ✅ Most powerful models (GPT-4, Claude, Gemini)                 │
│  ├─ ✅ Zero hardware requirements                                   │
│  ├─ ✅ Always up-to-date                                            │
│  ├─ ❌ Data leaves your control                                     │
│  ├─ ❌ Ongoing costs ($0.01-$0.10+ per 1K tokens)                   │
│  └─ ❌ Requires internet connection                                 │
│                                                                     │
│  🖥️ LOCAL Models (Llama, Mistral, Phi)                              │
│  ├─ ✅ Complete data privacy                                        │
│  ├─ ✅ No per-request costs                                         │
│  ├─ ✅ Works offline                                                │
│  ├─ ✅ Full customization & fine-tuning                             │
│  ├─ ❌ Requires capable hardware                                    │
│  └─ ❌ You manage updates & security                                │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### When to Choose Local Models

| Use Case | Recommendation | Why |
|----------|----------------|-----|
| Healthcare/Medical | 🟢 **Local** | HIPAA compliance, patient data |
| Legal Documents | 🟢 **Local** | Client confidentiality |
| Enterprise Internal | 🟢 **Local** | Trade secrets, IP protection |
| Public Chatbots | 🟡 **Cloud** | Scale, reliability |
| Learning/Prototyping | 🟡 **Either** | Depends on budget |
| Air-gapped Systems | 🟢 **Local Only** | No internet access |

## 1.2 Model Formats & Quantization Explained

### Understanding Model Sizes

```
📦 ORIGINAL MODEL (Full Precision - FP32)
   └─ Llama 2 7B = ~28 GB in memory
   └─ Llama 2 13B = ~52 GB in memory
   └─ Llama 2 70B = ~280 GB in memory
   
🗜️ QUANTIZED MODEL (Reduced Precision)
   └─ Llama 2 7B Q4_K_M = ~4.4 GB (runs on 8GB RAM!)
   └─ Llama 2 7B Q8_0 = ~7.7 GB (better quality, more RAM)
```

### Quantization Formats Decoded

| Format | Bits | Size Reduction | Quality Loss | Best For |
|--------|------|----------------|--------------|----------|
| FP32 | 32-bit | Baseline | None | Training |
| FP16 | 16-bit | 50% | Minimal | GPU inference |
| Q8_0 | 8-bit | 75% | Very small | Quality-focused |
| Q6_K | 6-bit | 81% | Small | Balance |
| Q5_K_M | 5-bit | 84% | Small | **Recommended** |
| Q4_K_M | 4-bit | 87% | Moderate | Memory-constrained |
| Q3_K_S | 3-bit | 90% | Noticeable | Very low RAM |
| Q2_K | 2-bit | 93% | Significant | Experiments only |

### The GGUF Format Revolution

```
📁 OLD: Multiple files, complex loading
   ├── config.json
   ├── tokenizer.json
   ├── pytorch_model.bin (or .safetensors)
   └── ... many more files

📁 NEW: Single GGUF file, everything included!
   └── model-Q4_K_M.gguf  ← Weights + Tokenizer + Config
```

> **💡 GGUF** = "GPT-Generated Unified Format" - The standard for local LLM deployment, used by Ollama, llama.cpp, LM Studio, and more.

## 1.3 Hardware Requirements Guide

### RAM Requirements by Model Size

| Model Size | Q4 Quant | Q8 Quant | FP16 | Recommended RAM |
|------------|----------|----------|------|------------------|
| 1B-3B | 1-2 GB | 2-3 GB | 4-6 GB | 8 GB |
| 7B | 4-5 GB | 7-8 GB | 14 GB | 16 GB |
| 13B | 8-9 GB | 14-15 GB | 26 GB | 32 GB |
| 30B | 18-20 GB | 32-35 GB | 60 GB | 64 GB |
| 70B | 40-45 GB | 75-80 GB | 140 GB | 128+ GB |

### GPU vs CPU Inference

```
🖥️ CPU-Only Inference
├─ ✅ Works on any computer
├─ ✅ No GPU drivers needed
├─ ⚠️ Slower (10-50 tokens/second for 7B)
└─ Best for: Development, occasional use

🎮 GPU-Accelerated Inference
├─ ✅ Much faster (100-200+ tokens/second)
├─ ✅ Can offload partially (CPU+GPU hybrid)
├─ ⚠️ Requires CUDA (NVIDIA) or ROCm (AMD)
└─ Best for: Production, real-time apps

🍎 Apple Silicon (M1/M2/M3)
├─ ✅ Excellent unified memory
├─ ✅ Metal acceleration built-in
├─ ✅ 80-150 tokens/second on M2 Pro
└─ Best for: Mac users (great experience!)
```

### Quick Hardware Check Cheatsheet

| Your Setup | Can Run | Recommended Model |
|------------|---------|-------------------|
| 8GB RAM, no GPU | ✅ Small models | Phi-3 Mini, Llama 3.2 1B |
| 16GB RAM, no GPU | ✅ Most 7B models | Mistral 7B Q4, Llama 3 8B Q4 |
| 16GB RAM + RTX 3060 | ✅ Fast 7B, slow 13B | Mistral 7B Q5, CodeLlama 13B Q4 |
| 32GB RAM + RTX 4080 | ✅ Fast 13B, most 30B | Llama 3 70B Q4 (partial) |
| M2 Pro 32GB | ✅ Fast 13B, good 30B | Mixtral 8x7B, CodeLlama 34B |

---

# 🦙 Section 2: Ollama Deep Dive

## 2.1 Installation & Setup

### Installing Ollama

```bash
# 🪟 Windows
# Download from: https://ollama.com/download/windows
# Run the installer, Ollama runs as a system service

# 🍎 macOS
# Download from: https://ollama.com/download/mac
# Or use Homebrew:
brew install ollama

# 🐧 Linux
curl -fsSL https://ollama.com/install.sh | sh

# 🐳 Docker
docker run -d -v ollama:/root/.ollama -p 11434:11434 --name ollama ollama/ollama
```

### Verify Installation

```bash
# Check Ollama is running
ollama --version

# List available models
ollama list

# Check server status
curl http://localhost:11434/api/tags
```

## 2.2 Model Management with Ollama

### Essential Ollama Commands

```bash
# 📥 Pull models from the library whichever you want
ollama run qwen3  # if any specific size required give it's size like qwen3:14b
ollama run gpt-oss      
ollama pull llama3.2           
ollama pull llama3.2:1b        
ollama pull llama3.2:3b        
ollama pull mistral            
ollama pull codellama          
ollama pull phi3               
ollama pull deepseek-coder     

# 📋 List downloaded models
ollama list

# 🔍 Show model details
ollama show qwen3
ollama show qwen3 --modelfile  # See full model configuration

# 🗑️ Remove models
ollama rm old-model

# 🔄 Update a model
ollama pull qwen3  # Re-pulling updates to latest
```

In [21]:
# 📦 Install Python Libraries for Ollama Integration
!pip install ollama requests httpx
print("✅ Libraries installed successfully!")

✅ Libraries installed successfully!


## 2.3 Python Integration: Three Methods

### Method 1: Official Ollama Python Library (Recommended)

In [ ]:
# 🐍 Method 1: Official Ollama Python Library
import ollama

def simple_generate(prompt: str, model: str = "lfm2.5-thinking:1.2b-q4_K_M") -> str:
    """Generate text using Ollama's simplest API."""
    response = ollama.generate(model=model, prompt=prompt)
    return response['response']

# Example usage
result = simple_generate("Explain machine learning in 3 sentences.")
print("📝 Response:")
print(result)

model='lfm2.5-thinking:1.2b-q4_K_M' created_at='2026-01-24T06:29:53.7129473Z' done=True done_reason='stop' total_duration=1041746100 load_duration=161556600 prompt_eval_count=18 prompt_eval_duration=173261400 eval_count=186 eval_duration=668979700 response="Okay, the user wants me to explain machine learning in three sentences. Let me start by breaking down what machine learning is. It's a subset of AI that uses data to learn patterns. Then, how to fit that into three sentences. First sentence: define ML as learning from data. Second: mention algorithms improving with data. Third: applications like predictions or recommendations. Wait, need to make sure each sentence is concise. Let me check: first sentence introduces ML's role in learning patterns. Second explains how it works through data. Third gives examples like recommendation systems. That should work. Make sure it's three sentences, clear and concise. Avoid jargon. Okay, that should do it.\n</think>\n\nMachine learning is a subs

In [27]:
# 💬 Chat-based Interaction (with conversation history)
import ollama
from typing import List, Dict

class OllamaChat:
    """A chat interface for Ollama with conversation history."""
    
    def __init__(self, model: str = "qwen3:8b", system_prompt: str = None):
        self.model = model
        self.messages: List[Dict[str, str]] = []
        if system_prompt:
            self.messages.append({"role": "system", "content": system_prompt})
    
    def chat(self, user_message: str) -> str:
        """Send a message and get a response, maintaining history."""
        self.messages.append({"role": "user", "content": user_message})
        response = ollama.chat(model=self.model, messages=self.messages)
        assistant_message = response['message']['content']
        self.messages.append({"role": "assistant", "content": assistant_message})
        return assistant_message
    
    def clear_history(self):
        """Clear conversation history (keeps system prompt)."""
        self.messages = [m for m in self.messages if m['role'] == 'system']

# 🎮 Demo: Multi-turn conversation
print("=" * 60)
print("🤖 Multi-Turn Conversation Demo")
print("=" * 60)

chat = OllamaChat(
    model="gpt-oss:20b",
    system_prompt="You are a helpful assistant. Keep responses concise."
)

print("\n👤 User: What is a Python decorator?")
print(f"🤖 Assistant: {chat.chat('What is a Python decorator?')}")

print("\n👤 User: Show me a simple example.")
print(f"🤖 Assistant: {chat.chat('Show me a simple example.')}")

🤖 Multi-Turn Conversation Demo

👤 User: What is a Python decorator?
🤖 Assistant: A **Python decorator** is a function (or callable) that takes another function or method, adds some behavior to it, and returns a new function (usually a wrapper). Decorators let you “decorate” or modify functions without changing their code.

```python
def my_decorator(func):
    def wrapper(*args, **kwargs):
        # pre‑action
        result = func(*args, **kwargs)
        # post‑action
        return result
    return wrapper

@my_decorator            # syntactic sugar for: my_func = my_decorator(my_func)
def my_func():
    pass
```

Key points:
- Decorators are applied with `@decorator_name` above the target function.
- They can add logging, authentication, caching, etc.
- They can also be used to create class decorators, method decorators, and parameterized decorators.

In short, a decorator is a higher‑order function that enhances or modifies other functions or methods.

👤 User: Show me a simple ex

In [29]:
# ⚡ Streaming Responses (for real-time output)
import ollama

def stream_generate(prompt: str, model: str = "qwen3:8b"):
    """Stream responses token by token for real-time display."""
    print(f"🤖 Streaming response...\n")
    print("-" * 50)
    
    full_response = ""
    stream = ollama.generate(model=model, prompt=prompt, stream=True)
    
    for chunk in stream:
        token = chunk['response']
        print(token, end='', flush=True)
        full_response += token
    
    print("\n" + "-" * 50)
    print(f"\n✅ Complete! ({len(full_response)} characters)")
    return full_response

stream_generate("Write a long poem.")

🤖 Streaming response...

--------------------------------------------------
**Elegy for the Eternal Horizon**  

The sea, a cradle of whispered tides,  
Where moonlight weaves its silver threads,  
Drifts on the breath of ancient winds,  
A lullaby for the world’s unspoken dreams.  
Its waves, like ghosts of forgotten songs,  
Trace patterns on the shore’s worn skin,  
A dance of salt and starlit foam,  
A hymn to time’s unyielding spin.  

The mountains rise, their bones of stone,  
Carved by the hands of centuries,  
Their peaks crowned with the fire of dawn,  
A silent psalm to gravity’s decree.  
They wear the scars of tempests’ wrath,  
Yet stand as sentinels of the sky,  
Their shadows stretching, vast and deep,  
To cradle the earth in quiet sighs.  

The forests hum with roots entwined,  
Where time is measured in the rustle of leaves,  
And every tree is a keeper of secrets,  
Their bark etched with the language of roots.  
The fox, with eyes like embers, slips  
Through under

'**Elegy for the Eternal Horizon**  \n\nThe sea, a cradle of whispered tides,  \nWhere moonlight weaves its silver threads,  \nDrifts on the breath of ancient winds,  \nA lullaby for the world’s unspoken dreams.  \nIts waves, like ghosts of forgotten songs,  \nTrace patterns on the shore’s worn skin,  \nA dance of salt and starlit foam,  \nA hymn to time’s unyielding spin.  \n\nThe mountains rise, their bones of stone,  \nCarved by the hands of centuries,  \nTheir peaks crowned with the fire of dawn,  \nA silent psalm to gravity’s decree.  \nThey wear the scars of tempests’ wrath,  \nYet stand as sentinels of the sky,  \nTheir shadows stretching, vast and deep,  \nTo cradle the earth in quiet sighs.  \n\nThe forests hum with roots entwined,  \nWhere time is measured in the rustle of leaves,  \nAnd every tree is a keeper of secrets,  \nTheir bark etched with the language of roots.  \nThe fox, with eyes like embers, slips  \nThrough underbrush where shadows play,  \nWhile owls carve thei

### Method 2: Direct REST API with Requests

In [30]:
# 🌐 Method 2: Direct REST API
import requests

class OllamaAPI:
    """Direct REST API client for Ollama."""
    
    def __init__(self, base_url: str = "http://localhost:11434"):
        self.base_url = base_url
    
    def generate(self, prompt: str, model: str = "qwen3:8b", options: dict = None) -> dict:
        payload = {"model": model, "prompt": prompt, "stream": False}
        if options:
            payload["options"] = options
        response = requests.post(f"{self.base_url}/api/generate", json=payload)
        return response.json()
    
    def list_models(self) -> list:
        response = requests.get(f"{self.base_url}/api/tags")
        return response.json().get('models', [])

# Demo
api = OllamaAPI()
print("📋 Available Models:")
try:
    for m in api.list_models():
        print(f"   • {m['name']} ({m.get('size', 0) / (1024**3):.2f} GB)")

    for model_name in ["gpt-oss:20b", "qwen3:8b"]:
        print(f"\n📝 Generating text with model: {model_name}")
        response = api.generate(prompt="What is reinforcement learning?", model=model_name)
        print("   Response:")
        print(f"   {response['response']}")
except:
    print("   ⚠️ Ollama not running. Start with: ollama serve")

📋 Available Models:
   • lfm2.5-thinking:1.2b-q4_K_M (0.68 GB)
   • gpt-oss:latest (12.85 GB)
   • lfm2.5-thinking:latest (0.68 GB)
   • qwen3:8b (4.87 GB)
   • gpt-oss:20b (12.85 GB)

📝 Generating text with model: gpt-oss:20b
   Response:
   ### Reinforcement Learning (RL) – A Quick Overview

| Concept | What it means in RL |
|--------|----------------------|
| **Agent** | The learner / decision‑maker (e.g., a robot, a game AI). |
| **Environment** | Everything the agent interacts with (the world, a simulation). |
| **State** | The agent’s current situation (e.g., the robot’s sensor readings). |
| **Action** | What the agent can do (move left/right, press a button). |
| **Policy** | A strategy that maps states to actions. |
| **Reward** | A scalar signal the agent receives after taking an action. |
| **Value** | Expected future reward starting from a state (or state‑action pair). |
| **Goal** | Maximize the **cumulative** reward over time. |

---

## 1. The Core Idea

Reinforcement le

---

# ⚡ Section 5: Performance Optimization

## 5.1 Key Optimization Strategies

| Strategy | Speed Gain | Memory Reduction |
|----------|------------|------------------|
| Quantization (Q4) | 2x | 4x |
| GPU Offloading | 10x | - |
| Batch Processing | 3-5x | - |
| KV Cache | 20% | - |
| Context Pruning | - | 50%+ |

## 5.2 Advanced Hyperparameter Tuning

### Understanding Generation Parameters

| Parameter | Range | Effect | Use Case |
|-----------|-------|--------|----------|
| **Temperature** | 0.0 - 2.0 | Controls randomness. 0=deterministic, 1=balanced, 2=very random | 0.0 for facts, 0.7 for creative, 1.5+ for brainstorming |
| **Top P (Nucleus Sampling)** | 0.0 - 1.0 | Cumulative probability threshold. Lower = more focused | 0.9 for coherent, 0.5 for more focused |
| **Top K** | 1 - 1000 | Only consider top K tokens | 40-50 typical, lower = deterministic |
| **Repeat Penalty** | 0.0 - 2.0 | Penalizes repetition. Higher = less repetition | 1.1-1.3 to reduce loops |
| **Num Predict** | 1 - 4096 | Max tokens to generate | Limit response length |
| **Num Context** | 128 - 32768 | Context window size | Larger = more memory but better context |

### The Temperature vs Top P vs Top K Guide

```
TEMPERATURE (Randomness/Creativity)
─────────────────────────────────────
0.0  ████ Completely deterministic → Best for Q&A, facts
0.3  ████ Very focused
0.7  ████ Balanced (default)
1.0  ████ Creative
1.5  ████ Very creative
2.0  ████ Chaotic

TOP P (Probability Threshold)
──────────────────────────────
1.0  ▓▓▓▓ Use all tokens
0.9  ▓▓▓░ 90% probability mass
0.5  ▓▓░░ 50% probability mass
0.1  ▓░░░ Very focused

TOP K (Token Count Limit)
──────────────────────────
1    ▓░░░ Only top token (deterministic)
10   ▓▓░░ Top 10 tokens
50   ▓▓▓░ Top 50 tokens (typical)
100  ▓▓▓▓ Top 100 tokens (diverse)
```


In [31]:
# 🎛️ Practical: Temperature Control
import ollama

def compare_temperatures():
    """Compare responses at different temperature settings."""
    
    prompt = "Tell me a creative story about a robot learning to paint."
    temperatures = [0.1, 0.5, 0.9, 1.5]
    
    print("=" * 70)
    print("🌡️ TEMPERATURE COMPARISON")
    print("=" * 70)
    
    for temp in temperatures:
        print(f"\n📊 Temperature: {temp}")
        print("-" * 70)
        
        response = ollama.generate(
            model="qwen3:8b",  # or your available model
            prompt=prompt,
            options={
                "temperature": temp,
                "num_predict": 80  # Limit length for comparison
            }
        )
        print(response['response'][:200] + "...")

# Uncomment to run (requires Ollama running):
# compare_temperatures()


In [32]:
# 🎚️ Advanced: Combined Hyperparameter Control
import ollama
from typing import Dict

class HyperparameterConfig:
    """Preset configurations for different use cases."""
    
    PRECISE = {
        "temperature": 0.2,
        "top_p": 0.7,
        "top_k": 20,
        "repeat_penalty": 1.2
    }
    
    BALANCED = {
        "temperature": 0.7,
        "top_p": 0.9,
        "top_k": 50,
        "repeat_penalty": 1.1
    }
    
    CREATIVE = {
        "temperature": 1.3,
        "top_p": 0.95,
        "top_k": 100,
        "repeat_penalty": 1.0
    }
    
    DETERMINISTIC = {
        "temperature": 0.0,
        "top_p": 1.0,
        "top_k": 1,
        "repeat_penalty": 1.5
    }

def generate_with_config(prompt: str, config: str, model: str = "qwen3:8b") -> str:
    """Generate using preset configurations."""
    
    config_dict = getattr(HyperparameterConfig, config.upper())
    
    print(f"\n🎯 Using {config.upper()} config:")
    print(f"   Temperature: {config_dict['temperature']}")
    print(f"   Top P: {config_dict['top_p']}")
    print(f"   Top K: {config_dict['top_k']}")
    print(f"   Repeat Penalty: {config_dict['repeat_penalty']}")
    print("-" * 50)
    
    response = ollama.generate(
        model=model,
        prompt=prompt,
        options=config_dict,
        stream=False
    )
    
    return response['response']

# 📝 Example usage:
print("=" * 60)
print("🎛️ HYPERPARAMETER PRESETS DEMONSTRATION")
print("=" * 60)

test_prompt = "What makes a good software engineer?"

for config_name in ["precise", "balanced", "creative", "deterministic"]:
    result = generate_with_config(test_prompt, config_name)
    print(result[:150] + "...\n")


🎛️ HYPERPARAMETER PRESETS DEMONSTRATION

🎯 Using PRECISE config:
   Temperature: 0.2
   Top P: 0.7
   Top K: 20
   Repeat Penalty: 1.2
--------------------------------------------------
A good software engineer is a blend of technical expertise, problem-solving skills, and soft skills that enable them to create high-quality software w...


🎯 Using BALANCED config:
   Temperature: 0.7
   Top P: 0.9
   Top K: 50
   Repeat Penalty: 1.1
--------------------------------------------------
A good software engineer is a combination of technical expertise, problem-solving skills, and soft skills that enable them to create reliable, efficie...


🎯 Using CREATIVE config:
   Temperature: 1.3
   Top P: 0.95
   Top K: 100
   Repeat Penalty: 1.0
--------------------------------------------------
A good software engineer is a blend of technical mastery, problem-solving acumen, and collaborative mindset. Here’s a structured breakdown of the key ...


🎯 Using DETERMINISTIC config:
   Temperature: 0.0
  

In [ ]:
# 🔬 Deep Dive: Individual Parameter Effects
import ollama
import requests

class ParameterExplorer:
    """Explore individual parameter effects systematically."""
    
    def __init__(self, model: str = "qwen3:8b", base_url: str = "http://localhost:11434"):
        self.model = model
        self.base_url = base_url
    
    def explore_top_k(self, prompt: str, values: list = [1, 10, 50, 100]):
        """Explore Top K parameter."""
        print("\n📊 TOP K EXPLORATION")
        print("=" * 60)
        print("Lower K = More deterministic | Higher K = More diverse\n")
        
        for k in values:
            response = requests.post(
                f"{self.base_url}/api/generate",
                json={
                    "model": self.model,
                    "prompt": prompt,
                    "stream": False,
                    "options": {
                        "top_k": k,
                        "num_predict": 50
                    }
                }
            ).json()
            
            print(f"🔹 Top K = {k:3d}: {response['response'][:80]}...")
    
    def explore_top_p(self, prompt: str, values: list = [0.5, 0.75, 0.9, 1.0]):
        """Explore Top P (Nucleus Sampling)."""
        print("\n📊 TOP P EXPLORATION (Nucleus Sampling)")
        print("=" * 60)
        print("Lower P = More focused | Higher P = More diverse\n")
        
        for p in values:
            response = requests.post(
                f"{self.base_url}/api/generate",
                json={
                    "model": self.model,
                    "prompt": prompt,
                    "stream": False,
                    "options": {
                        "top_p": p,
                        "num_predict": 50
                    }
                }
            ).json()
            
            print(f"🔹 Top P = {p:.2f}: {response['response'][:80]}...")
    
    def explore_repeat_penalty(self, prompt: str, values: list = [0.8, 1.0, 1.2, 1.5]):
        """Explore Repeat Penalty."""
        print("\n📊 REPEAT PENALTY EXPLORATION")
        print("=" * 60)
        print("Lower = More repetition allowed | Higher = Avoid repetition\n")
        
        for penalty in values:
            response = requests.post(
                f"{self.base_url}/api/generate",
                json={
                    "model": self.model,
                    "prompt": prompt,
                    "stream": False,
                    "options": {
                        "repeat_penalty": penalty,
                        "num_predict": 50
                    }
                }
            ).json()
            
            print(f"🔹 Penalty = {penalty:.1f}: {response['response'][:80]}...")

# 🎯 Usage example:
try:
    explorer = ParameterExplorer(model="qwen3:8b")
    test_prompt = "List five programming languages:"
    
    # Uncomment to explore individual parameters:
    # explorer.explore_top_k(test_prompt)
    # explorer.explore_top_p(test_prompt)
    # explorer.explore_repeat_penalty(test_prompt)
    
    print("✅ Explorer ready! Uncomment methods above to run exploration.")
except Exception as e:
    print(f"⚠️ Error: {e}")


In [ ]:
# 🎯 Real-World Use Cases with Optimized Parameters

use_cases = {
    "Q&A (Factual)": {
        "description": "Answering factual questions where accuracy is critical",
        "config": {
            "temperature": 0.1,
            "top_p": 0.7,
            "top_k": 20,
            "repeat_penalty": 1.3,
            "num_predict": 200
        },
        "example_prompt": "What is the capital of France?"
    },
    "Content Writing": {
        "description": "Creating engaging blog posts or marketing copy",
        "config": {
            "temperature": 0.9,
            "top_p": 0.92,
            "top_k": 60,
            "repeat_penalty": 1.1,
            "num_predict": 500
        },
        "example_prompt": "Write a compelling introduction to a blog post about AI..."
    },
    "Code Generation": {
        "description": "Generating functional, reliable code",
        "config": {
            "temperature": 0.3,
            "top_p": 0.8,
            "top_k": 40,
            "repeat_penalty": 1.2,
            "num_predict": 300
        },
        "example_prompt": "Write a Python function to calculate fibonacci numbers:"
    },
    "Brainstorming": {
        "description": "Generating multiple creative ideas",
        "config": {
            "temperature": 1.4,
            "top_p": 0.95,
            "top_k": 100,
            "repeat_penalty": 1.0,
            "num_predict": 250
        },
        "example_prompt": "List 10 creative startup ideas for..."
    },
    "Dialogue/Roleplay": {
        "description": "Natural conversation with personality",
        "config": {
            "temperature": 1.1,
            "top_p": 0.93,
            "top_k": 80,
            "repeat_penalty": 1.15,
            "num_predict": 150
        },
        "example_prompt": "I want you to act as a helpful assistant..."
    }
}

print("=" * 80)
print("🎯 REAL-WORLD USE CASES & OPTIMIZED PARAMETERS")
print("=" * 80)

for use_case, details in use_cases.items():
    print(f"\n📌 {use_case}")
    print(f"   Description: {details['description']}")
    print("   Recommended Parameters:")
    for param, value in details['config'].items():
        print(f"      • {param:20} = {value}")


In [ ]:
# 🚀 Advanced: Dynamic Parameter Adjustment

class AdaptiveGenerator:
    """Dynamically adjust parameters based on input characteristics."""
    
    def __init__(self, model: str = "qwen3:8b"):
        self.model = model
    
    def detect_query_type(self, prompt: str) -> str:
        """Detect the type of query to auto-select parameters."""
        
        prompt_lower = prompt.lower()
        
        # Factual keywords
        if any(word in prompt_lower for word in ["what", "who", "where", "when", "how many", "define"]):
            return "factual"
        
        # Creative keywords
        elif any(word in prompt_lower for word in ["create", "write", "imagine", "story", "poem", "creative"]):
            return "creative"
        
        # Code keywords
        elif any(word in prompt_lower for word in ["code", "function", "script", "program", "write in", "python", "javascript"]):
            return "code"
        
        # Brainstorm keywords
        elif any(word in prompt_lower for word in ["ideas", "brainstorm", "suggest", "list", "examples"]):
            return "brainstorm"
        
        else:
            return "balanced"
    
    def get_auto_config(self, query_type: str) -> dict:
        """Get optimal config for detected query type."""
        
        configs = {
            "factual": {"temperature": 0.1, "top_p": 0.7, "top_k": 20, "repeat_penalty": 1.3},
            "creative": {"temperature": 1.0, "top_p": 0.92, "top_k": 70, "repeat_penalty": 1.1},
            "code": {"temperature": 0.3, "top_p": 0.8, "top_k": 40, "repeat_penalty": 1.2},
            "brainstorm": {"temperature": 1.3, "top_p": 0.95, "top_k": 100, "repeat_penalty": 1.0},
            "balanced": {"temperature": 0.7, "top_p": 0.9, "top_k": 50, "repeat_penalty": 1.1}
        }
        
        return configs.get(query_type, configs["balanced"])
    
    def generate_adaptive(self, prompt: str, show_config: bool = True) -> str:
        """Generate with automatically selected parameters."""
        
        query_type = self.detect_query_type(prompt)
        config = self.get_auto_config(query_type)
        
        if show_config:
            print(f"🤖 Detected Query Type: {query_type.upper()}")
            print(f"📊 Auto-Selected Config: {config}")
            print("-" * 60)
        
        response = ollama.generate(
            model=self.model,
            prompt=prompt,
            options=config,
            stream=False
        )
        
        return response['response']

# 🧪 Demo: Adaptive generation
print("=" * 60)
print("🚀 ADAPTIVE PARAMETER SELECTION")
print("=" * 60)

generator = AdaptiveGenerator()

test_prompts = [
    "What is photosynthesis?",
    "Write a creative poem about the ocean",
    "Write a Python function to sort a list",
    "Brainstorm ideas for a sustainable startup"
]

for prompt in test_prompts:
    print(f"\n❓ Prompt: {prompt}")
    try:
        result = generator.generate_adaptive(prompt)
        print(f"✅ {result[:100]}...\n")
    except:
        print("⚠️ Note: Run Ollama to execute. Here's what the system would do:")
        query_type = generator.detect_query_type(prompt)
        config = generator.get_auto_config(query_type)
        print(f"   Detected: {query_type} | Config: {config}\n")
